# Lexical Decision Task - İngilizce vs Türkçe Analizi
Bu notebook, **Trial 1 (İngilizce)** ve **Trial 2 (Türkçe)** arasındaki Tepki Süresi (RT) ve Doğruluk oranlarını karşılaştırır.

*Not: PsychoPy verisindeki cevap formatı hatası (örn: `left'` vs `left`) nedeniyle doğruluk sütunları sıfır görünüyordu. Bu notebook, doğru cevapları `keys` ve `corr_ans`/`cevap` sütunlarını eşleştirerek manuel ve kusursuz bir şekilde hesaplar.*

In [ ]:
import pandas as pd
import glob
import numpy as np
from scipy import stats

## 1. Veri Okuma ve Doğruluk (Accuracy) Düzeltmesi

In [ ]:
dosyalar = glob.glob("*.csv")
sonuclar = []

for dosya in dosyalar:
    if "Sonuclar" in dosya or "Analizi" in dosya:
        continue
        
    try:
        df = pd.read_csv(dosya, low_memory=False)
        participant = df['participant'].dropna().iloc[0] if 'participant' in df.columns else dosya.split('_')[0]
        
        # Sütun İsimleri
        rt1_col = 'key_resp_3.rt' if 'key_resp_3.rt' in df.columns else ('trials.key_resp_3.rt' if 'trials.key_resp_3.rt' in df.columns else None)
        keys1_col = 'key_resp_3.keys' if 'key_resp_3.keys' in df.columns else ('trials.key_resp_3.keys' if 'trials.key_resp_3.keys' in df.columns else None)
        corr_ans1 = 'corr_ans'
        
        rt2_col = 'key_resp_4.rt' if 'key_resp_4.rt' in df.columns else ('trials_2.key_resp_4.rt' if 'trials_2.key_resp_4.rt' in df.columns else None)
        keys2_col = 'key_resp_4.keys' if 'key_resp_4.keys' in df.columns else ('trials_2.key_resp_4.keys' if 'trials_2.key_resp_4.keys' in df.columns else None)
        corr_ans2 = 'cevap'
        
        # Trial 1 (İngilizce) Hesaplamaları
        df[rt1_col] = pd.to_numeric(df[rt1_col], errors='coerce')
        t1_rt = df[rt1_col].mean()
        
        df[corr_ans1] = df[corr_ans1].astype(str).str.replace("'", "", regex=False)
        t1_trials = df.dropna(subset=[keys1_col])
        t1_correct = (t1_trials[keys1_col].astype(str) == t1_trials[corr_ans1]).sum()
        
        # Trial 2 (Türkçe) Hesaplamaları
        df[rt2_col] = pd.to_numeric(df[rt2_col], errors='coerce')
        t2_rt = df[rt2_col].mean()
        
        if corr_ans2 in df.columns:
            df[corr_ans2] = df[corr_ans2].astype(str).str.replace("'", "", regex=False)
            t2_trials = df.dropna(subset=[keys2_col])
            t2_correct = (t2_trials[keys2_col].astype(str) == t2_trials[corr_ans2]).sum()
        else:
            t2_correct = np.nan
            
        sonuclar.append({
            'Dosya': dosya,
            'Katilimci': participant,
            'T1_RT_Eng': t1_rt,
            'T1_Dogru_Eng': t1_correct,
            'T2_RT_Tr': t2_rt,
            'T2_Dogru_Tr': t2_correct,
        })
    except Exception as e:
        pass

df_sonuc = pd.DataFrame(sonuclar).dropna()
display(df_sonuc)

## 2. Hipotez Testleri (Bağımlı Örneklemler T-Testi)
Beklenti: Türkçe (Trial 2) tepki süresinin daha kısa (RT_Tr < RT_Eng) ve doğru sayısının daha yüksek (Dogru_Tr > Dogru_Eng) olmasıdır.

In [ ]:
print("Ortalama Tepki Süresi (İngilizce):", df_sonuc['T1_RT_Eng'].mean())
print("Ortalama Tepki Süresi (Türkçe):", df_sonuc['T2_RT_Tr'].mean())
print("Ortalama Doğru Sayısı (İngilizce):", df_sonuc['T1_Dogru_Eng'].mean())
print("Ortalama Doğru Sayısı (Türkçe):", df_sonuc['T2_Dogru_Tr'].mean())

# Yönlü (One-Tailed) T-Testi
print("\n--- HİPOTEZ 1: Türkçe RT daha düşüktür ---")
t_rt, p_rt = stats.ttest_rel(df_sonuc['T1_RT_Eng'], df_sonuc['T2_RT_Tr'], alternative='greater')
print(f"t = {t_rt:.4f}, p = {p_rt:.4e}")
if p_rt < 0.05:
    print("Sonuç: p < 0.05. Hipotez DOĞRULANDI (Anlamlı fark var).")
else:
    print("Sonuç: p >= 0.05. Anlamlı fark bulunamadı.")

print("\n--- HİPOTEZ 2: Türkçe Doğru Sayısı daha yüksektir ---")
t_corr, p_corr = stats.ttest_rel(df_sonuc['T2_Dogru_Tr'], df_sonuc['T1_Dogru_Eng'], alternative='greater')
print(f"t = {t_corr:.4f}, p = {p_corr:.4e}")
if p_corr < 0.05:
    print("Sonuç: p < 0.05. Hipotez DOĞRULANDI (Anlamlı fark var).")
else:
    print("Sonuç: p >= 0.05. Anlamlı fark bulunamadı.")

## 3. Korelasyon (Hız - Doğruluk)

In [ ]:
r_t1, p_t1 = stats.pearsonr(df_sonuc['T1_RT_Eng'], df_sonuc['T1_Dogru_Eng'])
r_t2, p_t2 = stats.pearsonr(df_sonuc['T2_RT_Tr'], df_sonuc['T2_Dogru_Tr'])

print(f"İngilizce Bloğu: r = {r_t1:.4f}, p = {p_t1:.4f}")
print(f"Türkçe Bloğu: r = {r_t2:.4f}, p = {p_t2:.4f}")
print("\nNot: p > 0.05 olduğu için hız ve doğruluk arasında istatistiksel olarak anlamlı bir ilişki bulunmamaktadır.")

## 4. Kaydetme

In [ ]:
df_sonuc.to_csv('Ing_vs_Tr_Dogrulanmis_Sonuclar.csv', index=False)
print('Düzeltilmiş sonuçlar başarıyla kaydedildi.')